# 학습 사진 결함 위치 표시 — 300장

파일: `lvlm_sft_train_location_annotation_colab.ipynb`

**GPU 불필요(일반 CPU). 위에서부터 실행 → 표시 화면에서 박스 표시 → 마지막 결과 저장 셀을 실행하세요.**

LoRA 학습에 쓴 결함 사진 300장입니다(누락 60, 녹 180, 새 둥지 60). dev240이나 gold 사진이 아니며, 둘 다 바꾸지 않습니다.
화면에는 사진과 **결함 종류**만 나옵니다. 모델 답이나 자동으로 만든 박스는 보이지 않습니다.

- 마우스로 끌어서 **결함을 감싸는 박스**를 그리세요. 결함이 여러 곳이면 각각 그립니다.
- 화면 위 빨간 줄에 이 사진의 결함 종류와 무엇을 감싸야 하는지 나옵니다.
- 위치를 정하기 어려우면 `위치 판단 불가`, 결함 라벨 자체가 이상하면 `라벨 이견`을 누르고 사유를 적으세요. 이 사진들은 위치 학습에서 빠집니다.
- 박스를 그리거나 지우면 Drive에 **초안이 자동 저장**됩니다. `저장 후 다음(Enter)`을 눌러야 확정입니다. `Z`는 마지막 박스 지우기입니다.
- 중간에 나가도 됩니다. `초안 저장됨` 표시를 확인하고 나가면, 다음에 같은 노트북을 실행했을 때 첫 미완료 사진부터 이어집니다.
- 같은 종류가 12장씩 묶여 나옵니다. 60장을 끝낼 때마다 다섯 종류가 같은 수만큼 채워집니다.

결과 파일: `sft_train_location_annotation_results.zip`. 다 못 끝내도 마지막 셀로 중간 백업을 만들 수 있습니다.

In [ ]:
RUN_FOLDER = "lvlm_insplad_results/sft_train_location_annotation_sft_v1"
SUP_ZIP_PATH = ""  # 자동 검색이 안 될 때만 Drive의 실제 ZIP 경로를 입력

In [ ]:
import base64,hashlib,io,json,sys,zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
SOURCE_SHA256='da18780eef5f0783d7be739bdcc4e49b1cb395e94320e9e376daf3a8a23238ca'
SOURCE_MANIFEST={'scripts/sft_train_location_annotation.py': '32aa44d55a0ca9fdee9ad9db2887df5bfa58a7151ca0cadd4ad07cf9aa06e7e9', 'scripts/dev_missing_location_annotation.py': '3ff033e86b871f7052ea10b05cd06158bce4774801799f44449861f6440ead20', 'scripts/sft_train_location_ui.html': '6c4417e1b88534e65742a50d25bf92315494faf9da4f9a974cc1fee71284ee1d', 'inputs/manifest.json': '82bcedbf69b4ebb79839ac4d1555b3cf095f10a87c12fa0849d7faca375a3993'}
SOURCE_BYTES=base64.b64decode('...')
assert hashlib.sha256(SOURCE_BYTES).hexdigest()==SOURCE_SHA256
CODE=Path('/content/sft_train_location_'+SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    assert set(archive.namelist())==set(SOURCE_MANIFEST)
    for name in archive.namelist():
        dest=CODE/name;data=archive.read(name)
        assert dest.resolve().is_relative_to(CODE.resolve())
        assert hashlib.sha256(data).hexdigest()==SOURCE_MANIFEST[name]
        if dest.exists() and dest.read_bytes()!=data:raise RuntimeError('내장 코드와 다른 파일입니다. 새 런타임을 사용하세요.')
        dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
sys.path.insert(0,str(CODE/'scripts'))
import sft_train_location_annotation as annotation
def find_zip():
    if SUP_ZIP_PATH:
        p=Path(SUP_ZIP_PATH)
        if not p.is_file():raise FileNotFoundError(str(p))
        return p
    for p in [DRIVE/'supervised_fault_classification.zip',DRIVE/'lvlm_insplad_data/supervised_fault_classification.zip']:
        if p.is_file() and p.stat().st_size==874753167:return p
    matches=[p for p in DRIVE.rglob('supervised_fault_classification.zip') if p.is_file() and p.stat().st_size==874753167]
    if not matches:raise FileNotFoundError('supervised_fault_classification.zip 원본을 Drive에서 찾지 못했습니다.')
    return sorted(matches)[0]
OUT=DRIVE/RUN_FOLDER
assert OUT.name.startswith(annotation.PREFIX)
manifest=annotation.read(CODE/'inputs/manifest.json')
print('사진 300장을 원본 ZIP에서 꺼내 확인합니다. 1~2분 걸릴 수 있습니다.')
session=annotation.prepare(manifest,find_zip(),OUT,cache=Path('/content/sft_train_location_images'),
    progress=lambda i,n:print(f'  사진 확인 {i}/{n}',flush=True))
(OUT/'source_bundle.zip').write_bytes(SOURCE_BYTES)
annotation.atomic(OUT/'source_manifest.json',SOURCE_MANIFEST)
print('원본 SHA 확인 완료. 저장/재개 상태:',session.progress())
print('저장 위치:',OUT/'annotations.json')

In [ ]:
# 이 화면에서 박스를 표시합니다. 박스 추가/삭제는 초안 저장, Enter는 확정 후 다음.
annotation.register_colab(session)

## 표시를 마친 뒤(또는 중간 백업할 때) 아래 셀 실행

화면에서 `저장 후 다음`으로 확정한 사진만 위치 정답으로 들어갑니다.
아래 셀은 Drive에 저장된 상태로 결과를 만들고, 종류별 표시 수와 **위치가 9칸 중 어디에 몰려 있는지**를 출력합니다.

In [ ]:
from google.colab import files
result,summary=session.export()
print('확정',summary['done'],'/',summary['total'],'· 박스',summary['boxed'],'· 판단 불가',summary['unlocatable'],'· 이견',summary['disputed'])
print('종류별 박스 수:',summary['boxed_by_defect'])
print('위치 분포(박스 중심이 놓인 칸):',summary['region_counts'])
print('가운데 칸만 정답인 사진:',summary['centre_only_images'],'/',summary['positive_spatial_support'])
if not summary['complete']:print('미확정 사진이 있습니다. 중간 백업이며, 표시를 끝낸 뒤 이 셀을 다시 실행하세요.')
print('ZIP:',result)
files.download(str(result))